In [0]:
from pyspark.sql import functions as F

BASE = "workspace.desafio_sicredi"

print("Ambiente de testes:", BASE)

Ambiente de testes: workspace.desafio_sicredi


In [0]:
df_transacoes = spark.table(
    f"{BASE}.silver_transacoes_cartao"
)

duplicados = (
    df_transacoes
    .groupBy("id_transacao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicados == 0, (
    f"Falha: existem {duplicados} IDs de transação duplicados."
)

print("BINGO — Teste 1: transações sem duplicidade.")

BINGO — Teste 1: transações sem duplicidade.


In [0]:
valores_invalidos = (
    df_transacoes
    .filter(
        F.col("valor").isNull() |
        (F.col("valor") <= 0)
    )
    .count()
)

assert valores_invalidos == 0, (
    f"Falha: existem {valores_invalidos} valores inválidos."
)

print("BINGO — Teste 2: valores financeiros válidos.")

BINGO — Teste 2: valores financeiros válidos.


In [0]:
df_cartoes = spark.table(
    f"{BASE}.silver_cartoes"
)

transacoes_orfas = (
    df_transacoes
    .join(
        df_cartoes.select("id_cartao").distinct(),
        on="id_cartao",
        how="left_anti"
    )
    .count()
)

assert transacoes_orfas == 0, (
    f"Falha: existem {transacoes_orfas} transações sem cartão válido."
)

print("BINGO — Teste 3: integridade referencial.")

BINGO — Teste 3: integridade referencial.


In [0]:
estornos_invalidos = (
    df_transacoes
    .filter(
        (F.col("estornada") == True) &
        (
            F.col("valor_liquido").isNull() |
            (F.col("valor_liquido") != 0)
        )
    )
    .count()
)

assert estornos_invalidos == 0, (
    f"Falha: existem {estornos_invalidos} estornos inconsistentes."
)

print("BINGO — Teste 4: tratamento de estornos.")

BINGO — Teste 4: tratamento de estornos.


In [0]:
df_gold = spark.table(
    f"{BASE}.gold_transacoes_cartao"
)

total_silver = (
    df_transacoes
    .agg(F.sum("valor_liquido").alias("total"))
    .first()["total"]
)

total_gold = (
    df_gold
    .agg(F.sum("valor_liquido").alias("total"))
    .first()["total"]
)

assert total_silver == total_gold, (
    f"Falha: Silver={total_silver}, Gold={total_gold}"
)

print("BINGO — Teste 5: conciliação financeira Silver x Gold.")

BINGO — Teste 5: conciliação financeira Silver x Gold.


In [0]:
duplicados_gold = (
    df_gold
    .groupBy("id_transacao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicados_gold == 0, (
    f"Falha: existem {duplicados_gold} transações duplicadas na Gold."
)

print("BINGO — Teste 6: unicidade das transações na Gold.")

BINGO — Teste 6: unicidade das transações na Gold.
